# Phân loại sách OK / NG trên Google Colab

Mỗi ảnh nhận một nhãn. **NG** là sách lỗi, **OK** là sách bình thường. Notebook fine-tune **YOLO11n-cls** từ trọng số ImageNet rồi xuất checkpoint Ultralytics dùng được với bộ đọc `probs` của trạm.

**Cách chạy.** Chọn *Runtime → Change runtime type → T4 GPU* (hoặc GPU đang có), rồi *Run all*. Khi hiện nút chọn tệp, tải lên `book-defect-v23.v1i.folder.zip` nguyên gói. Nếu ZIP đã có trên Google Drive, điền đường dẫn tại ô bên dưới. Bật `SAVE_TO_DRIVE` nếu muốn giữ checkpoint khi Colab ngắt phiên.

| Tập trong ZIP đã kiểm | NG | OK | Tổng |
|---|---:|---:|---:|
| train | 76 | 61 | 137 |
| valid | 19 | 20 | 39 |
| test | 15 | 4 | 19 |
| **Tổng** | **110** | **85** | **195** |

Ảnh gốc có 172 ảnh dọc và 23 ảnh ngang. Ảnh xem trước cho thấy nhiều sách được chụp hai mặt. ZIP thiếu mã cuốn nên kết quả mặc định đo trên **split ảnh Roboflow**. Muốn đánh giá trên **cuốn/thiết kế bìa chưa từng gặp**, điền bảng nhóm ở mục 2. Với split gốc, một sách tốt bị báo lỗi làm tỷ lệ nhận đúng lớp OK trên test thay đổi **25 điểm phần trăm**.

Nguồn bộ ảnh ghi trong ZIP là *book-defect-v23*, bản 1, CC BY 4.0. [Dataset Roboflow](https://universe.roboflow.com/nghia-nguyen-gajwm/book-defect-v23) · [Ultralytics Classification](https://docs.ultralytics.com/tasks/classify/)

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/content/book-defect-v23.v1i.folder.zip")
SAVE_TO_DRIVE = False
DRIVE_OUTPUT = Path("/content/drive/MyDrive/PBL5/book-defect-v23")
GROUPS_CSV = None  # Có thể điền "/content/drive/MyDrive/PBL5/book_groups.csv".

IMAGE_SIZE = 448  # configs/station.json
EPOCHS = 60
PATIENCE = 12
BATCH = 16  # Giảm xuống 8 nếu GPU báo hết bộ nhớ.
SEED = 42

## 1. Chuẩn bị môi trường và chọn tệp

Code tự tìm ZIP ở đường dẫn đã khai báo trước khi mở cửa sổ tải lên. Mỗi lượt chạy đủ notebook tạo một thư mục kết quả riêng. Colab cần Internet để tải thư viện và trọng số pretrained lần đầu.

In [ ]:
import importlib.metadata
import subprocess
import sys

ULTRALYTICS_VERSION = "8.4.208"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       f"ultralytics=={ULTRALYTICS_VERSION}"])

import csv
import hashlib
import json
import tempfile
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations
from pathlib import PurePosixPath
from zipfile import ZipFile, ZIP_DEFLATED

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import ultralytics
from google.colab import drive, files
from PIL import Image
from ultralytics import YOLO

if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError("Hãy khởi động lại runtime để dùng phiên bản Ultralytics vừa cài.")
drive_root = Path("/content/drive")
inputs = [ZIP_PATH] + ([Path(GROUPS_CSV)] if GROUPS_CSV else [])
if SAVE_TO_DRIVE or any(p.is_relative_to(drive_root) for p in inputs):
    drive.mount(str(drive_root))
if not ZIP_PATH.is_file():
    if ZIP_PATH.is_relative_to(drive_root):
        raise FileNotFoundError(f"Kiểm tra lại đường dẫn ZIP trên Drive: {ZIP_PATH}")
    print("Chọn một tệp ZIP của bộ ảnh.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy chọn đúng một tệp ZIP, rồi chạy lại ô này.")
    ZIP_PATH = Path(next(iter(uploaded))).resolve()
    del uploaded

WORK = Path(tempfile.mkdtemp(prefix="book-v23-", dir=Path.cwd()))
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
OUTPUT = (DRIVE_OUTPUT if SAVE_TO_DRIVE else WORK) / run_id
OUTPUT.mkdir(parents=True)
REPORT = OUTPUT / "report"
REPORT.mkdir()
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Thiết bị:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (có thể đổi sang GPU để train nhanh hơn)")
print("Torch", torch.__version__, "| Ultralytics", ultralytics.__version__, "| Kết quả", OUTPUT)

## 2. Kiểm tra ảnh, split và nhóm sách

Mặc định giữ các tập do Roboflow cung cấp. Nếu có CSV với hai cột `archive_path,book_id`, đặt `GROUPS_CSV` ở ô cấu hình. Gán cùng một mã cho mọi ảnh của cùng cuốn/thiết kế bìa, kể cả mặt sau và các lần chụp khác. CSV cần đủ một dòng cho từng ảnh. Notebook xuất mẫu `book_groups_template.csv` trong thư mục báo cáo.

Với CSV đã điền, cả nhóm vào **test** nếu nhóm có ảnh test gốc, kế đến ưu tiên **val**, còn lại vào **train**. Cách này giữ riêng mọi ảnh test gốc. Bảng đếm được tính lại sau khi gom. Notebook dừng nếu thiếu lớp hoặc ảnh trùng hệt sau tiền xử lý xuất hiện ở nhiều tập.

Ảnh được đệm xám mức 114 và thu về 448 × 448 bằng OpenCV giống hàm `square_cover` của trạm. Toàn bộ mép bìa được giữ lại. Tệp PNG đã xử lý là đầu vào của cả train và đánh giá.

In [ ]:
prepared = Path(tempfile.mkdtemp(prefix="split-", dir=WORK))
DATA = prepared / "dataset"
HOLDOUT = prepared / "test"
LABELS = ("NG", "OK")
SPLITS = ("train", "val", "test")
split_names = {"train": "train", "valid": "val", "val": "val", "test": "test"}
known_sha = "91e6cbcef0c8208a0a6607f53749f1cbcaa2110578f9ec298f791eb77f998cb7"
with ZIP_PATH.open("rb") as f:
    archive_sha = hashlib.file_digest(f, "sha256").hexdigest()
print("SHA-256:", archive_sha, "|", "đúng ZIP đã phân tích" if archive_sha == known_sha else "ZIP khác bản đã phân tích")

def write_csv(path, fields, rows):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def square_cover(image):
    h, w = image.shape[:2]
    side = max(h, w)
    top, left = (side - h) // 2, (side - w) // 2
    canvas = cv2.copyMakeBorder(image, top, side-h-top, left, side-w-left,
                                cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return cv2.resize(canvas, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

def image_path(row):
    base = HOLDOUT if row["split"] == "test" else DATA / row["split"]
    return base / row["label"] / row["file"]

rows = []
with ZipFile(ZIP_PATH) as archive:
    for info in sorted(archive.infolist(), key=lambda member: member.filename):
        path = PurePosixPath(info.filename)
        if info.is_dir() or (len(path.parts) == 1 and path.suffix.lower() == ".txt"):
            continue
        if (len(path.parts) != 3 or path.parts[0] not in split_names or
                path.parts[1] not in LABELS or path.suffix.lower() not in {".jpg", ".jpeg", ".png"}):
            raise ValueError(f"Mục ngoài cấu trúc split/nhãn/ảnh: {path}")
        split = split_names[path.parts[0]]
        rows.append(dict(archive_path=info.filename, source_split=split, split=split,
                         label=path.parts[1], book_id="", file=f"img_{len(rows):04d}.png"))

    archive_paths = {row["archive_path"] for row in rows}
    if not rows or len(archive_paths) != len(rows):
        raise ValueError("ZIP rỗng hoặc có đường dẫn ảnh trùng nhau.")
    write_csv(REPORT / "book_groups_template.csv", ["archive_path", "book_id"], rows)
    if GROUPS_CSV:
        with Path(GROUPS_CSV).open(encoding="utf-8-sig", newline="") as f:
            entries = list(csv.DictReader(f))
        groups = {entry["archive_path"]: entry["book_id"].strip() for entry in entries}
        if set(groups) != archive_paths or len(entries) != len(groups) or not all(groups.values()):
            raise ValueError("CSV nhóm phải khớp đủ các ảnh, mỗi ảnh đúng một mã cuốn.")
        group_splits = defaultdict(set)
        for row in rows:
            row["book_id"] = groups[row["archive_path"]]
            group_splits[row["book_id"]].add(row["source_split"])
        for row in rows:
            row["split"] = max(group_splits[row["book_id"]], key=SPLITS.index)
    counts = Counter((r["split"], r["label"]) for r in rows)
    for split in SPLITS:
        print(split, {label: counts[split, label] for label in LABELS})
        if any(counts[split, label] == 0 for label in LABELS):
            raise ValueError(f"Tập {split} thiếu lớp. Kiểm tra lại dữ liệu và bảng nhóm.")

    seen = {}
    for row in rows:
        raw = archive.read(row["archive_path"])
        image = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Không đọc được {row['archive_path']}")
        row["height"], row["width"] = image.shape[:2]
        square = square_cover(image)
        key = hashlib.sha256(square.tobytes()).hexdigest()
        if key in seen and (seen[key]["label"], seen[key]["split"]) != (row["label"], row["split"]):
            raise ValueError(f"Ảnh trùng khác nhãn/tập: {seen[key]['archive_path']} và {row['archive_path']}")
        seen[key] = row
        target = image_path(row)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not cv2.imwrite(str(target), square):
            raise OSError(f"Lỗi ghi ảnh {target}")
        row["sha256_input"] = key
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        small = np.asarray(Image.fromarray(gray).resize((9, 8), Image.Resampling.LANCZOS))
        row["dhash"] = int.from_bytes(np.packbits(small[:, :-1] > small[:, 1:]).tobytes(), "big")

write_csv(REPORT / "manifest.csv", ["archive_path", "source_split", "split", "label", "book_id",
                                    "file", "width", "height", "sha256_input"], rows)
candidates = []
for a, b in combinations(rows, 2):
    if a["split"] != b["split"]:
        distance = (a["dhash"] ^ b["dhash"]).bit_count()
        if distance <= 4:
            candidates.append(dict(first=a["archive_path"], second=b["archive_path"], distance=distance))
write_csv(REPORT / "near_pairs.csv", ["first", "second", "distance"], candidates)
print("Đã đọc", len(rows), "ảnh. Số cặp khác tập cần xem theo dHash:", len(candidates))
print("dHash chỉ gợi ý để rà ảnh gần giống. Hai mặt của cùng cuốn có thể có dHash rất khác nhau.")

In [ ]:
import random

rng = random.Random(SEED)
fig, axes = plt.subplots(2, 4, figsize=(10, 5), squeeze=False)
for axis_row, label in zip(axes, LABELS):
    pool = [r for r in rows if r["split"] == "train" and r["label"] == label]
    for ax in axis_row:
        ax.axis("off")
    for ax, row in zip(axis_row, rng.sample(pool, min(4, len(pool)))):
        with Image.open(image_path(row)) as img:
            ax.imshow(img)
        ax.set_title(f"{label} · {row['file']}")
plt.tight_layout()
plt.show()

## 3. Fine-tune YOLO11n-cls

Lượt baseline dùng tối đa 60 epoch, dừng sớm sau 12 epoch nếu điểm validation không cải thiện. Dữ liệu ít nên bắt đầu từ checkpoint pretrained. Chỉ thay đổi màu và độ sáng nhẹ khi train. Giữ tỷ lệ ảnh, tắt lật, xóa vùng ngẫu nhiên và crop thu nhỏ để tránh mất dấu lỗi ở góc bìa.

Ultralytics chọn `best.pt` theo fitness trên validation. Với hai lớp, top-5 luôn chứa nhãn đúng, nên thứ tự checkpoint theo fitness phụ thuộc vào top-1. Ảnh test nằm ở thư mục riêng ngoài dataset mà trainer đọc.

In [ ]:
model = YOLO("yolo11n-cls.pt")
if model.task != "classify":
    raise ValueError(f"Checkpoint sai tác vụ: {model.task}")

model.train(
    data=str(DATA), project=str(OUTPUT), name="train", device=DEVICE,
    imgsz=IMAGE_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=2,
    pretrained=True, optimizer="AdamW", lr0=0.0005, weight_decay=0.0005,
    scale=0.0, fliplr=0.0, flipud=0.0, erasing=0.0, auto_augment=None,
    hsv_h=0.01, hsv_s=0.10, hsv_v=0.12, seed=SEED, deterministic=True,
    cache=False, plots=True, save=True,
)
RUN = Path(model.trainer.save_dir)
BEST = RUN / "weights" / "best.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"Chưa tạo được checkpoint {BEST}")
print("Checkpoint được chọn bằng validation:", BEST)

## 4. Đọc lại best.pt và đo trên test

Mỗi ảnh được phân loại theo xác suất cao nhất, giống chế độ Direct của trạm. **NG → OK** là bỏ sót sách lỗi. **OK → NG** là loại nhầm sách tốt. Xem cả tử số và mẫu số khi đọc tỷ lệ. Nếu đang so nhiều cấu hình, dùng validation để chọn cấu hình rồi chỉ chạy mục test khi đã chốt.

In [ ]:
best = YOLO(str(BEST))
if best.task != "classify" or set(best.names.values()) != set(LABELS):
    raise ValueError(f"Checkpoint không khớp hai lớp: {best.names}")
class_id = {label: idx for idx, label in best.names.items()}
cm = np.zeros((2, 2), dtype=int)  # Hàng = nhãn thật, cột = dự đoán. Thứ tự NG, OK.
predictions = []
for row in (r for r in rows if r["split"] == "test"):
    result = best.predict(str(image_path(row)), imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)[0]
    prob = result.probs
    if prob is None:
        raise ValueError("Model không trả xác suất phân loại.")
    pred = best.names[prob.top1]
    truth = row["label"]
    cm[LABELS.index(truth), LABELS.index(pred)] += 1
    predictions.append(dict(archive_path=row["archive_path"], truth=truth, prediction=pred,
                            p_NG=float(prob.data[class_id["NG"]]), p_OK=float(prob.data[class_id["OK"]])))

write_csv(REPORT / "test_predictions.csv", ["archive_path", "truth", "prediction", "p_NG", "p_OK"], predictions)
by_class = {}
for i, label in enumerate(LABELS):
    tp, actual, predicted = int(cm[i, i]), int(cm[i].sum()), int(cm[:, i].sum())
    by_class[label] = dict(support=actual, correct=tp, recall=tp/actual,
                           precision=tp/predicted if predicted else None, f1=2*tp/(actual+predicted))
metrics = dict(n=int(cm.sum()), accuracy=float(cm.trace()/cm.sum()),
               balanced_accuracy=sum(m["recall"] for m in by_class.values())/len(LABELS),
               macro_f1=sum(m["f1"] for m in by_class.values())/len(LABELS),
               by_class=by_class, label_order=list(LABELS), confusion_true_by_pred=cm.tolist())
print(f"Accuracy {cm.trace()}/{cm.sum()} = {metrics['accuracy']:.1%}  |  "
      f"Balanced accuracy {metrics['balanced_accuracy']:.1%}  |  Macro F1 {metrics['macro_f1']:.3f}")
print(f"Bỏ sót NG → OK: {cm[0, 1]}/{cm[0].sum()}  |  Loại nhầm OK → NG: {cm[1, 0]}/{cm[1].sum()}")
for label, m in by_class.items():
    prec = "N/A" if m["precision"] is None else f"{m['precision']:.1%}"
    print(f"{label}: recall {m['correct']}/{m['support']} = {m['recall']:.1%}, precision {prec}, F1 {m['f1']:.3f}")

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(1, cm.max()))
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center", color="white" if value > cm.max()/2 else "black")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=LABELS, yticklabels=LABELS,
       xlabel="Dự đoán", ylabel="Nhãn thật", title="Tập test • số ảnh")
fig.tight_layout()
fig.savefig(REPORT / "test_confusion.png", dpi=160)
plt.show()

## 5. Tải model và báo cáo

Gói ZIP chứa `book_defect_v23_best.pt`, log huấn luyện, danh sách ảnh theo split, xác suất từng ảnh test, ma trận nhầm lẫn và các chỉ số. Nếu lưu trên Drive, notebook in đường dẫn bản đã lưu. Nếu dùng bộ nhớ Colab, trình duyệt tải gói về máy ở cuối ô.

Khi thử riêng model mới trên PC, đặt file vào `weights/book_defect_v23_best.pt`. Model mới trả các tên lớp `NG` và `OK`. Tệp `station_model_fields.json` trong gói liệt kê **ba trường** cần cập nhật trong bản cấu hình dùng thử của trạm. Model hiện tại dùng hai tên lớp tiếng Việt. Với chế độ hiệu chuẩn, cần đo lại ngưỡng theo model mới. Dùng sách có nhãn thật và chưa thấy lúc train để kiểm tra riêng trên camera băng tải.

In [ ]:
def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

with BEST.open("rb") as f:
    weight_sha = hashlib.file_digest(f, "sha256").hexdigest()
summary = dict(dataset=ZIP_PATH.name, archive_sha256=archive_sha, checkpoint_sha256=weight_sha,
               split_protocol="book_groups" if GROUPS_CSV else "roboflow_image_split",
               class_counts={s: {k: counts[s, k] for k in LABELS} for s in SPLITS},
               near_pairs=len(candidates), image_size=IMAGE_SIZE, seed=SEED, model_names=best.names,
               pretrained="yolo11n-cls.pt", python=sys.version.split()[0],
               libraries={p: importlib.metadata.version(p) for p in ("ultralytics", "torch", "torchvision")},
               device=torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu", test=metrics)
write_json(REPORT / "summary.json", summary)
weight_name = "book_defect_v23_best.pt"
write_json(REPORT / "station_model_fields.json",
           dict(model="weights/" + weight_name, classes={label: label for label in LABELS}, image_size=IMAGE_SIZE))

bundle = OUTPUT / f"book_defect_v23_{run_id}.zip"
with ZipFile(bundle, "w", ZIP_DEFLATED) as archive:
    archive.write(BEST, weight_name)
    for path in sorted(REPORT.iterdir()):
        if path.is_file():
            archive.write(path, "report/" + path.name)
    for name in ("args.yaml", "results.csv", "results.png"):
        if (RUN / name).is_file():
            archive.write(RUN / name, "train/" + name)
with ZipFile(bundle) as archive:
    if archive.testzip() is not None:
        raise OSError("Lỗi CRC trong gói kết quả.")
print(f"Gói kết quả ({bundle.stat().st_size / 2**20:.1f} MiB): {bundle}")
if not SAVE_TO_DRIVE:
    files.download(str(bundle))